# Tmux Tools Tests

This notebook tests the dialoghelper **tmux** module, which provides functions to capture and inspect content from tmux sessions, windows, and panes.

## Key Functions

- `shell_ret(cmd)` — Run shell commands locally or over SSH
- `pane(n, pane, session, window)` — Capture scrollback history from a tmux pane
- `list_panes()` / `panes()` — List or capture all panes in a window
- `list_windows()` / `windows()` — List or capture all windows in a session
- `list_sessions()` / `sessions()` — List or capture all tmux sessions
- `flatten_dict(d)` — Flatten nested dicts into `(path, value)` tuples
- `set_default_history(n)` — Set default scrollback line count

**Prerequisites:** tmux must be installed (`brew install tmux` on macOS).

**Run each cell in order.** Cell 2 creates a test tmux session automatically.

In [ ]:
# Cell 1: Imports
from dialoghelper.tmux import (
    shell_ret, pane, list_panes, panes,
    list_windows, windows, list_sessions,
    sessions, flatten_dict, set_default_history
)

print('Tmux imports ready!')

In [ ]:
# Cell 2: Create a test tmux session with some content
import subprocess, time

SESSION = 'dialeng_test'

# Kill existing test session if any
subprocess.run(f'tmux kill-session -t {SESSION} 2>/dev/null', shell=True)

# Create session with two windows
subprocess.run(f'tmux new-session -d -s {SESSION} -n main -x 120 -y 30', shell=True, check=True)
subprocess.run(f'tmux new-window -t {SESSION} -n logs', shell=True, check=True)

# Send commands to window 0 (main)
subprocess.run(f"tmux send-keys -t {SESSION}:main 'echo Hello from main window' Enter", shell=True)
time.sleep(0.3)
subprocess.run(f"tmux send-keys -t {SESSION}:main 'echo Current dir: $(pwd)' Enter", shell=True)
time.sleep(0.3)

# Send commands to window 1 (logs)
subprocess.run(f"tmux send-keys -t {SESSION}:logs 'echo Log output line 1' Enter", shell=True)
time.sleep(0.3)
subprocess.run(f"tmux send-keys -t {SESSION}:logs 'echo Log output line 2' Enter", shell=True)
time.sleep(0.3)

print(f'Test session "{SESSION}" created with 2 windows (main, logs)')

In [ ]:
# Cell 3: shell_ret — run shell commands
result = shell_ret('echo "shell_ret works!"')
print(f'Result: {result!r}')

# Can also capture multi-line output
result2 = shell_ret('echo line1; echo line2; echo line3')
print(f'Multi-line: {result2!r}')

In [ ]:
# Cell 4: list_sessions — show all tmux sessions
print('=== All Sessions ===')
print(list_sessions())

In [ ]:
# Cell 5: list_windows — show windows in our test session
print(f'=== Windows in {SESSION} ===')
print(list_windows(session=SESSION))

In [ ]:
# Cell 6: list_panes — show panes in a specific window
print(f'=== Panes in {SESSION}:main ===')
print(list_panes(session=SESSION, window=0))

print(f'\n=== Panes in {SESSION}:logs ===')
print(list_panes(session=SESSION, window=1))

In [ ]:
# Cell 7: pane — capture scrollback from a specific pane
print('=== Main window pane (last 10 lines) ===')
content = pane(n=10, session=SESSION, window=0)
print(content)

print('\n=== Logs window pane (last 10 lines) ===')
content2 = pane(n=10, session=SESSION, window=1)
print(content2)

In [ ]:
# Cell 8: panes — capture all panes as a dict
d = panes(session=SESSION, window=0, n=5)
print(f'Type: {type(d)}')
print(f'Pane keys: {list(d.keys())}')
for pane_num, content in d.items():
    print(f'\nPane {pane_num}:')
    print(content[:200])

In [ ]:
# Cell 9: windows — capture all windows and their panes as nested dict
w = windows(session=SESSION, n=5)
print(f'Type: {type(w)}')
print(f'Window keys: {list(w.keys())}')
for win_name, pane_dict in w.items():
    print(f'\nWindow "{win_name}": {len(pane_dict)} pane(s)')
    for pane_num, content in pane_dict.items():
        preview = content.strip()[:80]
        print(f'  Pane {pane_num}: {preview!r}...')

In [ ]:
# Cell 10: sessions — capture everything (all sessions, windows, panes)
s = sessions(n=5)
print(f'Type: {type(s)}')
print(f'Session keys: {list(s.keys())}')

# Show structure
for sess_name, win_dict in s.items():
    print(f'\nSession "{sess_name}": {len(win_dict)} window(s)')
    for win_name, pane_dict in win_dict.items():
        print(f'  Window "{win_name}": {len(pane_dict)} pane(s)')

In [ ]:
# Cell 11: flatten_dict — flatten nested session data for searching
flat = flatten_dict(s)
print(f'Got {len(flat)} flattened entries\n')
for path, content in flat:
    preview = content.strip()[:60].replace('\n', ' ')
    print(f'{path}: {preview!r}...')

In [ ]:
# Cell 12: set_default_history — change default scrollback capture
from dialoghelper.tmux import default_tmux_lines
print(f'Default lines before: {default_tmux_lines}')

set_default_history(100)

from dialoghelper import tmux
print(f'Default lines after: {tmux.default_tmux_lines}')

# Capture with new default (100 lines instead of 500)
content = pane(session=SESSION, window=0)
line_count = len(content.split('\n'))
print(f'Captured {line_count} lines with new default')

# Reset to original
set_default_history(500)
print(f'Reset to {tmux.default_tmux_lines}')

In [ ]:
# Cell 13: Practical example — search across all tmux content
flat = flatten_dict(sessions(n=20))

# Search for a keyword across all sessions/windows/panes
keyword = 'Hello'
matches = [(path, content) for path, content in flat if keyword in content]
print(f'Found "{keyword}" in {len(matches)} pane(s):')
for path, content in matches:
    # Find the matching lines
    lines = [l for l in content.split('\n') if keyword in l]
    print(f'  {path}:')
    for line in lines:
        print(f'    {line.strip()}')

In [ ]:
# Cell 14: Cleanup — kill the test session
import subprocess
subprocess.run(f'tmux kill-session -t {SESSION}', shell=True)
print(f'Test session "{SESSION}" cleaned up')

# Verify it's gone
remaining = shell_ret('tmux list-sessions 2>&1')
print(f'Remaining sessions: {remaining.strip() or "(none)"}')

## Summary

If all cells ran successfully, you've verified:

- **shell_ret()** — Run shell commands and capture output
- **list_sessions() / list_windows() / list_panes()** — Enumerate tmux hierarchy
- **pane()** — Capture scrollback history from a specific pane
- **panes() / windows() / sessions()** — Capture content as nested dicts
- **flatten_dict()** — Flatten nested dicts for searching across all panes
- **set_default_history()** — Configure default scrollback line count

### How It Works

```
sessions()                          # All sessions
  └─ windows(session=...)           # All windows in a session
       └─ panes(session=..., window=...)  # All panes in a window
            └─ pane(session=..., window=..., pane=...)  # Single pane content

list_sessions() / list_windows() / list_panes()  # Metadata only (no content)

flatten_dict(sessions())  # Flat list of (path, content) for searching
```

### Use Cases

- **LLM tool calling** — All capture functions are `@llmtool` decorated, so they can be used as AI assistant tools
- **Debugging** — Search across all terminal output for errors or specific strings
- **Monitoring** — Capture output from long-running processes in other panes
- **SSH support** — All functions accept `host` or `ip/user/keyfile` params for remote tmux

### Requirements

- tmux installed (`brew install tmux` on macOS)
- `dialoghelper` package with tmux module